In [3]:
import numpy as np
import pickle
import pandas as pd
import plotly.graph_objects as go

# Load necessary data
with open('../../../data/raw/aus_region_mask.pkl','rb') as f:
    lsmdf = pickle.load(f)
with open('../../../data/raw/aus_region_mask_lsmc.pkl','rb') as f:
    lsmc = pickle.load(f)
with open('../../../data/raw/wind_correlation_matrix.pkl','rb') as f:
    rlsmcs5Wmnavg = pickle.load(f)

In [4]:
# Print types and shapes
print(f"Type of lsmdf: {type(lsmdf)}")
print(f"Type of lsmc: {type(lsmc)}, Shape: {lsmc.shape}")
print(f"Type of rlsmcs5Wmnavg: {type(rlsmcs5Wmnavg)}, Shape: {rlsmcs5Wmnavg.shape}")

# Inspect the content if necessary
print(lsmdf.head())  

Type of lsmdf: <class 'xarray.core.dataset.Dataset'>
Type of lsmc: <class 'numpy.ndarray'>, Shape: (134, 166)
Type of rlsmcs5Wmnavg: <class 'numpy.ndarray'>, Shape: (11822, 11822)
<xarray.Dataset> Size: 708B
Dimensions:     (valid_time: 5, latitude: 5, longitude: 5)
Coordinates:
    number      int64 8B 0
  * valid_time  (valid_time) datetime64[ns] 40B 1999-01-01 ... 1999-01-01T04:...
  * latitude    (latitude) float64 40B -10.5 -10.75 -11.0 -11.25 -11.5
  * longitude   (longitude) float64 40B 112.8 113.0 113.2 113.5 113.8
    expver      (valid_time) <U4 80B ...
Data variables:
    lsm         (valid_time, latitude, longitude) float32 500B nan nan ... 0.0
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-18T02:42 GRIB to CDM+CF v

In [3]:
# Calculate mean correlations
mean_correlations = np.mean(rlsmcs5Wmnavg, axis=1)
mean_correlation_map = np.empty(np.shape(lsmc)) * np.nan
land_indices = np.argwhere(np.ndarray.flatten(lsmc) == 1)
land_coords = np.unravel_index(land_indices.flatten(), np.shape(lsmc))
mean_correlation_map[land_coords] = mean_correlations

# Prepare data for plotting
longitude = lsmdf.longitude.values
latitude = lsmdf.latitude.values

# Flatten the data for use in scattermapbox
mean_correlations_flat = np.ndarray.flatten(mean_correlation_map)
mean_correlations_land = mean_correlations_flat[land_indices.flatten()]

df = pd.DataFrame({
    'Latitude': latitude[land_coords[0]],
    'Longitude': longitude[land_coords[1]],
    'Mean Correlation': mean_correlations_land
})

In [4]:
# Print shapes of all key elements
print(f"Shape of lsmc (land-sea mask): {lsmc.shape}")  # (134, 166)
print(f"Shape of rlsmcs5Wmnavg (wind correlation matrix): {rlsmcs5Wmnavg.shape}")  # (11822, 11822)

print(f"Shape of mean_correlations (computed from correlation matrix): {mean_correlations.shape}")  # (11822,)

print(f"Shape of mean_correlation_map (mapped correlation grid): {mean_correlation_map.shape}")  # (134, 166)

print(f"Shape of land_indices (indices of land points in lsmc): {land_indices.shape}")  # (11822, 1)
print(f"Shape of land_coords (unraveled land coordinates in lsmc): {len(land_coords)} -> {land_coords[0].shape}, {land_coords[1].shape}")  # Two arrays

# Print longitude and latitude shapes
print(f"Shape of longitude: {longitude.shape}")  # Check how many longitude points
print(f"Shape of latitude: {latitude.shape}")  # Check how many latitude points

Shape of lsmc (land-sea mask): (134, 166)
Shape of rlsmcs5Wmnavg (wind correlation matrix): (11822, 11822)
Shape of mean_correlations (computed from correlation matrix): (11822,)
Shape of mean_correlation_map (mapped correlation grid): (134, 166)
Shape of land_indices (indices of land points in lsmc): (11822, 1)
Shape of land_coords (unraveled land coordinates in lsmc): 2 -> (11822,), (11822,)
Shape of longitude: (166,)
Shape of latitude: (134,)


In [9]:
target_lat_1, target_lon_1 = -30.0, 135.0  # Example
target_lat_2, target_lon_2 = -28.0, 133.0  # Example

# Find the closest index in latitude and longitude
index_1 = np.argmin(np.abs(latitude - target_lat_1))  # Find closest latitude index
index_2 = np.argmin(np.abs(latitude - target_lat_2))
index_3 = np.argmin(np.abs(longitude - target_lon_1))  # Find closest longitude index
index_4 = np.argmin(np.abs(longitude - target_lon_2))

# Find the index in land_coords that matches (index_1, index_3)
land_index_1 = np.where((land_coords[0] == index_1) & (land_coords[1] == index_3))[0][0]
land_index_2 = np.where((land_coords[0] == index_2) & (land_coords[1] == index_4))[0][0]

# Get the correlation
pairwise_correlation = rlsmcs5Wmnavg[land_index_1, land_index_2]

print(f"Pairwise correlation between ({target_lat_1}, {target_lon_1}) and ({target_lat_2}, {target_lon_2}): {pairwise_correlation}")

Pairwise correlation between (-30.0, 135.0) and (-28.0, 133.0): 0.7517194151878357


In [19]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Target location
target_lat, target_lon = -37.75, 147

# Find closest indices in latitude & longitude arrays
lat_index = np.argmin(np.abs(latitude - target_lat))  # Closest latitude
lon_index = np.argmin(np.abs(longitude - target_lon))  # Closest longitude

# Find corresponding land index
land_index = np.where((land_coords[0] == lat_index) & (land_coords[1] == lon_index))[0]

if len(land_index) == 0:
    print(f"No land found at ({target_lat}, {target_lon}). Try adjusting the coordinates slightly.")
else:
    land_index = land_index[0]  # Get the first match

    # Extract correlation values for all locations against the chosen point
    correlation_values = rlsmcs5Wmnavg[land_index, :]

    # Create a 2D correlation map
    correlation_map = np.empty(lsmc.shape) * np.nan
    correlation_map[land_coords] = correlation_values

    # Plot using Plotly for an interactive map
    df = pd.DataFrame({
        'Latitude': latitude[land_coords[0]],
        'Longitude': longitude[land_coords[1]],
        'Correlation': correlation_values
    })

    fig = go.Figure(go.Scattermap(
    lat=df['Latitude'],
    lon=df['Longitude'],
    mode='markers',
    marker=dict(
        size=5,  # Adjust marker size as needed
        color=df['Correlation'],
        colorscale='RdBu',  # Choose your color scale
        colorbar=dict(title='Mean Correlation'),
        opacity=0.8
    ),
    # text=df['Mean Correlation'],  # Optional: Add text labels
))

    fig.update_layout(
        title='Mean Wind Correlation Across Australia',
        mapbox=dict(
            style="open-street-map",  # Choose a map style
            center=dict(lat=-25, lon=135),  # Center the map on Australia
            zoom=3  # Adjust zoom level
        ),
        margin=dict(l=0, r=0, t=40, b=0)  # Adjust margins
    )
    
    fig.show()

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [ ]:

# Create and show the map
fig = go.Figure(go.Scattermap(
    lat=df['Latitude'],
    lon=df['Longitude'],
    mode='markers',
    marker=dict(
        size=5,  # Adjust marker size as needed
        color=df['Mean Correlation'],
        colorscale='RdBu',  # Choose your color scale
        colorbar=dict(title='Mean Correlation'),
        opacity=0.8
    ),
    # text=df['Mean Correlation'],  # Optional: Add text labels
))

fig.update_layout(
    title='Mean Wind Correlation Across Australia',
    mapbox=dict(
        style="open-street-map",  # Choose a map style
        center=dict(lat=-25, lon=135),  # Center the map on Australia
        zoom=3  # Adjust zoom level
    ),
    margin=dict(l=0, r=0, t=40, b=0)  # Adjust margins
)

fig.write_image("fig1.png")